In [1]:
!git clone https://github.com/nlihin/R2R-analysis

Cloning into 'R2R-analysis'...
remote: Enumerating objects: 380, done.
remote: Counting objects: 100% (380/380), done.
remote: Compressing objects: 100% (351/351), done.
remote: Total 380 (delta 118), reused 124 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (380/380), 280.15 KiB | 6.51 MiB/s, done.
Resolving deltas: 100% (118/118), done.


# Data Loading from Repository

In [2]:
import os
import pandas as pd
import numpy as np
from scipy.stats import linregress

BASE_PATH = "R2R-analysis/data"
CASE_STUDIES = ["case_study_1", "case_study_2", "case_study_3"]
DATA_TYPES = ["ratings", "IOL", "more_qs"]


def extract_session_number(filename):
    digits = "".join(filter(str.isdigit, filename))
    return int(digits) if digits else None


def load_case_study_files(base_path, case_studies, data_type, min_session=None):
    all_dataframes = []
    reference_columns = None

    for case_study in case_studies:
        folder_path = os.path.join(base_path, case_study, data_type)
        print(f"Processing folder: {folder_path}")

        if not os.path.exists(folder_path):
            print(f"Folder not found: {folder_path}")
            continue

        file_list = sorted([f for f in os.listdir(folder_path) if f.endswith(".csv")])

        for i, filename in enumerate(file_list):
            session_number = extract_session_number(filename)

            if min_session is not None and session_number is not None:
                if session_number < min_session:
                    print(f"Skipping file: {filename}")
                    continue

            file_path = os.path.join(folder_path, filename)

            if reference_columns is None and not all_dataframes:
                print(f"Reading first file with headers: {file_path}")
                df = pd.read_csv(file_path)
                reference_columns = df.columns
            else:
                print(f"Reading file without headers: {file_path}")
                df = pd.read_csv(file_path, skiprows=1, header=None)
                df.columns = reference_columns

            df.iloc[:, 0] = df.iloc[:, 0].astype(str) + f"_{session_number}"
            df["session_number"] = session_number
            all_dataframes.append(df)

    if not all_dataframes:
        return pd.DataFrame()

    return pd.concat(all_dataframes, ignore_index=True)


ratings = load_case_study_files(BASE_PATH, CASE_STUDIES, "ratings", min_session=7)
iol = load_case_study_files(BASE_PATH, CASE_STUDIES, "IOL")
more_qs = load_case_study_files(BASE_PATH, CASE_STUDIES, "more_qs")


Processing folder: R2R-analysis/data/case_study_1/ratings
Reading first file with headers: R2R-analysis/data/case_study_1/ratings/10.csv
Skipping file: 1_rate.csv
Skipping file: 2_rate.csv
Skipping file: 3_rate.csv
Skipping file: 4_rate.csv
Skipping file: 5_rate.csv
Skipping file: 6_rate.csv
Reading file without headers: R2R-analysis/data/case_study_1/ratings/7_rate.csv
Reading file without headers: R2R-analysis/data/case_study_1/ratings/8_rate.csv
Reading file without headers: R2R-analysis/data/case_study_1/ratings/9_rate.csv
Processing folder: R2R-analysis/data/case_study_2/ratings
Reading file without headers: R2R-analysis/data/case_study_2/ratings/11_rate.csv
Reading file without headers: R2R-analysis/data/case_study_2/ratings/12_rate.csv
Reading file without headers: R2R-analysis/data/case_study_2/ratings/13_rate.csv
Reading file without headers: R2R-analysis/data/case_study_2/ratings/14_rate.csv
Reading file without headers: R2R-analysis/data/case_study_2/ratings/15_rate.csv
Read

In [3]:
unique_usernames_count = ratings["username"].nunique()
print("Number of unique usernames in ratings from session 7 and above:", unique_usernames_count)

Number of unique usernames in ratings from session 7 and above: 270


In [4]:
more_qs.head()

,username,question_number,answer,group_number,session_number
0,uid12_10,1,3,3,10
1,uid12_10,2,4,3,10
2,uid12_10,3,3,3,10
3,uid14_10,1,4,3,10
4,uid14_10,2,5,3,10


In [5]:
more_qs[more_qs['question_number']==3]

,username,question_number,answer,group_number,session_number
2,uid12_10,3,3,3,10
5,uid14_10,3,4,3,10
8,uid2_10,3,3,3,10
11,uid15_10,3,1,3,10
14,uid17_10,3,3,3,10
...,...,...,...,...,...
7139,uid3_20,3,3,5,20
7142,uid1_20,3,5,6,20
7145,uid7_20,3,5,6,20
7148,uid3_20,3,5,6,20


## Rating-pattern feature calculation

The calculations in this section are intentionally kept identical to the original notebook logic.

In [6]:
check = ratings[ratings["username"] == "uid104_3"].copy()
check.info()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   username        0 non-null      object
 1   group_number    0 non-null      int64 
 2   rate            0 non-null      int64 
 3   feedback        0 non-null      object
 4   time            0 non-null      object
 5   session_number  0 non-null      int64 
dtypes: int64(3), object(3)
memory usage: 0.0+ bytes


In [7]:
import numpy as np
import pandas as pd
from scipy.stats import linregress


def split_mean_difference(x, split_ratio=0.5):
    x = x.values
    split_index = int(len(x) * split_ratio)
    if split_index == 0 or split_index == len(x):
        return np.nan
    return x[:split_index].mean() - x[split_index:].mean()


def calculate_pattern_stats(
    df,
    value_col,
    username_col="username",
    group_col="group_number",
    time_col=None,
    prefix=""
):
    data = df.copy()

    if time_col is not None:
        data["timestamp"] = pd.to_datetime(data[time_col])

    data["original_username"] = data[username_col]
    data[["uid", "session"]] = data[username_col].str.split("_", expand=True)
    data["session"] = data["session"].astype(int)

    data["normalized_rank"] = (
        data.groupby(["session", group_col])[value_col]
        .rank(method="min", ascending=True)
        / data.groupby(["session", group_col])[value_col].transform("count")
    )

    patrn9_name = f"{prefix}patrn9"
    user_avg_rank = (
        data.groupby("original_username")["normalized_rank"]
        .mean()
        .reset_index()
        .rename(columns={"normalized_rank": patrn9_name})
    )

    agg_dict = {
        f"{prefix}patrn1": (value_col, lambda x: np.std(x, ddof=1)),
        f"{prefix}patrn2": (value_col, "mean"),
        f"{prefix}patrn3": (value_col, "median"),
        f"{prefix}patrn4": (value_col, lambda x: np.sum(x >= 5) / len(x) if len(x) > 0 else 0),
        f"{prefix}patrn5": (value_col, lambda x: linregress(range(len(x)), x).slope if len(x) > 1 else np.nan),
        f"{prefix}patrn7": (value_col, lambda x: split_mean_difference(x, 0.5)),
        f"{prefix}patrn8": (value_col, lambda x: split_mean_difference(x, 0.7)),
        "num_ratings": (value_col, "count")
    }

    if time_col is not None:
        agg_dict[f"{prefix}patrn6"] = (
            "timestamp",
            lambda x: x.diff().mean().total_seconds() if len(x) > 1 else np.nan
        )

    user_stats = data.groupby("original_username").agg(**agg_dict).reset_index()

    user_stats[f"{prefix}patrn10"] = (
        user_stats[f"{prefix}patrn7"] / user_stats["num_ratings"]
    ).apply(lambda x: 1 if x > 0 else 0)

    user_stats[f"{prefix}patrn11"] = (
        user_stats[f"{prefix}patrn8"] / user_stats["num_ratings"]
    ).apply(lambda x: 1 if x > 0 else 0)

    final_stats = user_stats.merge(user_avg_rank, on="original_username")
    final_stats = final_stats.rename(columns={"original_username": username_col})
    final_stats = final_stats.drop(columns=["num_ratings"])

    return final_stats

In [8]:
ratings_stats = calculate_pattern_stats(
    df=ratings,
    value_col="rate",
    time_col="time",
    prefix=""
)

/tmp/ipykernel_667/1030428982.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["timestamp"] = pd.to_datetime(data[time_col])


In [9]:
understand_project_stats = calculate_pattern_stats(
    df=more_qs[more_qs['question_number']==3],
    value_col="answer",
    time_col=None,
    prefix="understand_project_"
)

## Reverse columns in questioner

In [10]:
cols_1to5 = ["satf5"]
cols_1to7 = ["fatig3", "fatig5"]
missing_1to5 = [c for c in cols_1to5 if c not in iol.columns]
missing_1to7 = [c for c in cols_1to7 if c not in iol.columns]
if missing_1to5 or missing_1to7:
    print("columns not found", {"1-5": missing_1to5, "1-7": missing_1to7})

for col in cols_1to5:
    if col in iol.columns:
        orig = iol[col].copy()
        v = pd.to_numeric(iol[col], errors="coerce")
        rev = 6 - v
        mask = v.between(1, 5)
        iol[col] = np.where(v.notna() & mask, rev, orig)

for col in cols_1to7:
    if col in iol.columns:
        orig = iol[col].copy()
        v = pd.to_numeric(iol[col], errors="coerce")
        rev = 8 - v
        mask = v.between(1, 7)
        iol[col] = np.where(v.notna() & mask, rev, orig)

## Merge the rating-pattern features with IOL


In [11]:
all_data = (
    iol
    .merge(ratings_stats, on="username", how="left")
    .merge(understand_project_stats, on="username", how="left")
)

In [15]:
all_data = all_data.dropna(subset=["patrn1"])
all_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 232 entries, 0 to 233
Data columns (total 45 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   username                    232 non-null    object 
 1   Duration (in seconds)       232 non-null    int64  
 2   age_bin                     232 non-null    object 
 3   gender                      232 non-null    object 
 4   satf1                       232 non-null    int64  
 5   satf2                       232 non-null    int64  
 6   satf3                       232 non-null    int64  
 7   satf4                       232 non-null    int64  
 8   satf5                       232 non-null    int64  
 9   satf6                       232 non-null    int64  
 10  iol1                        232 non-null    int64  
 11  iol2                        232 non-null    int64  
 12  iol3                        232 non-null    int64  
 13  iol4                        232 non-null

In [18]:
all_data.isna().any(axis=1).sum()

np.int64(0)

In [16]:
duplicate_usernames = all_data[all_data.duplicated(subset=["username"], keep=False)]
print(f"Number of rows with duplicate usernames: {duplicate_usernames.shape[0]}")

if not duplicate_usernames.empty:
    print("Duplicate usernames:")
    print(duplicate_usernames["username"].value_counts())
else:
    print("There are no duplicate usernames.")

Number of rows with duplicate usernames: 0
There are no duplicate usernames.


## Save the final dataset

In [17]:
all_data.to_csv("all_data_to_sem.csv", index=False)